# Deux usages, deux critères — et ce que coûte un plancher de rappel

Ce carnet rejoue la section 8.3 et la note de provenance
[`fusion_tstat_osu.PROVENANCE.md`](../data/results/fusion_tstat_osu.PROVENANCE.md).

Il répond à la question opérationnelle : **si l'on veut retrouver 90 % des
bâtiments détruits, combien d'adresses faut-il aller voir, et quelle est la
densité de détruits dans cette liste ?**

Table employée : `fusion_tstat_osu.csv`. Trois précisions la commandent, et le
carnet les vérifie avant d'en tirer un chiffre : le dénominateur, la définition
du positif, et le sort des bâtiments sans score partenaire.

In [ ]:
import os
import pandas as pd

# les carnets se lancent depuis notebooks/ ; les tables sont dans data/results
RESULTATS = os.path.join("..", "data", "results")
TABLES = os.path.join("..", "tables")

def lire(dossier, nom, **kw):
    """Lit une table du depot. Le separateur et la virgule decimale varient
    d'une campagne a l'autre : on laisse pandas les deviner sauf indication."""
    kw.setdefault("sep", None)
    kw.setdefault("engine", "python")
    return pd.read_csv(os.path.join(dossier, nom), **kw)

def presque(a, b, tol=0.0006):
    """Egalite au millieme pres, tolerance par defaut d'un demi-millieme."""
    assert abs(a - b) < tol, "%.4f attendu, %.4f obtenu" % (b, a)
    return True


Le dénominateur est le même partout ; les positifs dépendent de la définition retenue.

In [ ]:
f = lire(RESULTATS, "fusion_tstat_osu.csv")
f["n"] = f.tp + f.fp + f.fn + f.tn
f["positifs"] = f.tp + f.fn
print("denominateur unique :", sorted(f.n.unique()))
print()
print(f.groupby("gt_variant")["positifs"].unique().to_string())

Trois définitions du positif, trois prévalences.

In [ ]:
# la definition du positif, et la prevalence correspondante
for gt in sorted(f.gt_variant.unique()):
    bloc = f[f.gt_variant == gt]
    pos = int(bloc.positifs.iloc[0])
    print("%-26s %6d positifs   prevalence %.3f %%"
          % (gt, pos, 100 * pos / bloc.n.iloc[0]))

Pour chaque plancher de rappel : la longueur de la liste, et la densité de détruits qu'on y trouve.

In [ ]:
# le cout d'un plancher de rappel, sur la definition la plus stricte
d = f[(f.gt_variant == "Destroyed_seul") &
      (f.type_resultat == "palier_rappel")].copy()
d["liste"] = d.tp + d.fp
d["part_pct"] = 100 * d.liste / d.n
d["un_sur"] = (d.liste / d.tp).round(0)
ordre = ["toutes_confidences", "high_probable_et_possible",
         "high_et_probable", "high_seul"]
d["rang"] = d.niveau_confiance_osu.apply(ordre.index)
print(d.sort_values(["rang", "rappel_cible"], ascending=[True, False])
      [["niveau_confiance_osu", "rappel_cible", "seuil_tstat", "liste",
        "part_pct", "recall", "precision", "un_sur"]]
      .round({"part_pct": 2, "recall": 3, "precision": 5})
      .to_string(index=False))

À rappel égal, la conjonction divise la liste par 2,3 et multiplie la densité par 2,3.

In [ ]:
# les deux chiffres cites dans l'article, a 90 % de rappel
q = d[(d.rappel_cible == 0.90)]
seul = q[q.niveau_confiance_osu == "toutes_confidences"].iloc[0]
fus = q[q.niveau_confiance_osu == "high_probable_et_possible"].iloc[0]
print("T-stat seul : %6d batiments (%.2f %%), 1 detruit sur %d"
      % (seul.liste, seul.part_pct, round(seul.liste / seul.tp)))
print("fusion      : %6d batiments (%.2f %%), 1 detruit sur %d"
      % (fus.liste, fus.part_pct, round(fus.liste / fus.tp)))
print("\nliste divisee par %.1f, densite multipliee par %.1f"
      % (seul.liste / fus.liste, fus.precision / seul.precision))
presque(seul.part_pct, 37.25, tol=0.01)
presque(fus.part_pct, 16.27, tol=0.01)

Le plafond de rappel, et la raison de ne pas le confondre avec un effet de couverture.

In [ ]:
# le plafond : ce que le terme partenaire laisse passer, quel que soit le seuil
atteints = (d.groupby("niveau_confiance_osu")["rappel_cible"]
            .max().reindex(ordre))
print("plancher de rappel le plus haut encore atteignable :")
print((100 * atteints).to_string())
print("\nCe plafond est une propriete de la conjonction, non un effet de")
print("couverture : la note de provenance le mesure a 55,4 / 81,6 / 91,6 / 100 %.")